# NVDA 原始 CFO Data Center 表格数值与 Stage1 可用日

只取同一 SEC 原件表格当前季和上年同期，不跨版本补值；保守下一 Qlib 日生效，不读取 SOXX 结果。
原执行在 Recorder 完成后遇到 NumPy 标量 JSON 序列化错误；后续仅从同一 Recorder 收尾，详见 output_finalize_plan.json。


In [ ]:
import os,sys,json,hashlib,re,datetime,math,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from bs4 import BeautifulSoup
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings
card=root/'research/stage1_nvda_datacenter_original_numeric_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_HTTP_GETs','market_outcome_reads','model_fits','native_accounts'))
parent=root/src['parent_card'];coverage=root/src['coverage_card'];assert sha(parent/'quarter_event_adjudication_result.json')==src['selected_events_sha256'] and sha(coverage/'summary.json')==src['coverage_summary_sha256']
assert json.loads((parent/'item202_inventory_result.json').read_text())['NAS_archive_sha256']==src['parent_raw_NAS_archive_sha256']
assert sha(root/src['Qlib_provider']/'qlib.yaml')==src['Qlib_config_sha256']
source_events=json.loads((parent/'quarter_event_adjudication_result.json').read_text())['selected_events'];events=[e for e in source_events if e['issuer']=='NVDA'];assert len(events)==src['selected_NVDA_events']==63
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_provider']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'));assert len(cal)==3966
rows=[]
for e in events:
 p=root/src['source_root']/src['raw_file_template'].format(accession=e['accession']);assert p.is_file() and sha(p)==e['raw_sha256']
 full=p.read_text(errors='replace');m=re.search(r'<ACCEPTANCE-DATETIME>\s*([0-9]{14})',full,re.I);assert m,(e['accession'],'acceptance')
 accepted=m.group(1);accepted_date=pd.Timestamp(datetime.datetime.strptime(accepted[:8],'%Y%m%d').date());assert str(accepted_date.date())==e['filing_date']
 next_idx=cal.searchsorted(accepted_date,side='right');assert next_idx<len(cal);effective=cal[next_idx]
 docs=[z.group(1) for z in re.finditer(r'<DOCUMENT>(.*?)</DOCUMENT>',full,re.I|re.S)]
 cfo=[z for z in docs if re.search(r'<TYPE>\s*EX-99\.2(?:\s|<)',z,re.I)]
 row={'announcement_year':e['announcement_year'],'slot':e['slot'],'accession':e['accession'],'raw_sha256':e['raw_sha256'],'filing_date':e['filing_date'],'accepted_EDGAR_clock':accepted,'first_conservative_Qlib_signal_date':str(effective.date()),'CFO_EX99_2_count':len(cfo),'status':'unknown','current_million':None,'prior_quarter_million':None,'prior_year_million':None,'same_release_YoY':None,'reported_YoY_pct':None,'reported_YoY_diff_pp':None,'original_row':None,'table_unit':'unknown','definition_warning_Mellanox':False}
 if len(cfo)==1:
  body=re.search(r'<TEXT>(.*?)</TEXT>',cfo[0],re.I|re.S)
  if body:
   soup=BeautifulSoup(body.group(1),'html.parser');text=' '.join(soup.stripped_strings);row['definition_warning_Mellanox']=bool(re.search('Mellanox',text,re.I))
   matches=[]
   for tab in soup.find_all('table'):
    tabtext=' '.join(tab.stripped_strings)
    if not re.search(r'Revenue\s+by\s+Market\s+Platform',tabtext,re.I):continue
    for tr in tab.find_all('tr'):
     cells=[' '.join(c.stripped_strings) for c in tr.find_all(['td','th'])]
     if cells and re.fullmatch(r'Data\s*Center|Datacenter',cells[0].strip(),re.I):matches.append((tabtext,cells))
   if len(matches)==1:
    tabtext,cells=matches[0];row['original_row']=cells;unit_ok=bool(re.search(r'\$\s*in\s*millions|\(\$\s*in\s*millions\)',tabtext,re.I));row['table_unit']='USD_millions' if unit_ok else 'unknown'
    def parse_num(x):
     m=re.fullmatch(r'\$?\s*([0-9][0-9,]*)',x.strip());return int(m.group(1).replace(',','')) if m else None
    nums=[parse_num(v) for v in cells[1:]];numeric=[x for x in nums if x is not None]
    if unit_ok and len(numeric)>=3 and min(numeric[:3])>0:
     cur,priorq,priory=numeric[:3];growth=cur/priory-1
     trailing=' '.join(cells[4:]);pct=[(x.group(1),float(x.group(2))) for x in re.finditer(r'(?:(Up|Down)\s*)?([0-9]+(?:\.[0-9]+)?)\s*%',trailing,re.I)]
     reported=(pct[-1][1]*(-1 if pct[-1][0] and pct[-1][0].lower()=='down' else 1)) if pct else None
     diff=abs(100*growth-reported) if reported is not None else None
     if diff is None or diff<=1.5:
      row.update(status='numeric_same_release_table',current_million=cur,prior_quarter_million=priorq,prior_year_million=priory,same_release_YoY=growth,reported_YoY_pct=reported,reported_YoY_diff_pp=diff)
     else:row['status']='reported_YoY_conflict'
    else:row['status']='row_present_numeric_or_unit_unknown'
   elif len(matches)>1:row['status']='ambiguous_multiple_Data_Center_rows'
   else:row['status']='no_exact_Data_Center_market_platform_row'
  else:row['status']='EX99_2_no_TEXT'
 else:row['status']='EX99_2_not_unique'
 rows.append(row)
assert len(rows)==63 and len({r['accession'] for r in rows})==63
numeric=[r for r in rows if r['status']=='numeric_same_release_table'];unknown=[r for r in rows if r['status']!='numeric_same_release_table']
events_by_date=sorted(rows,key=lambda r:r['first_conservative_Qlib_signal_date']);assert len({r['first_conservative_Qlib_signal_date'] for r in rows})==63
feature=[];ages=[];event_accession=[];event_date=[];j=0;latest=None
for d in cal:
 while j<len(events_by_date) and pd.Timestamp(events_by_date[j]['first_conservative_Qlib_signal_date'])<=d:
  latest=events_by_date[j];j+=1
 feature.append(latest['same_release_YoY'] if latest and latest['status']=='numeric_same_release_table' else np.nan)
 ages.append((d-pd.Timestamp(latest['first_conservative_Qlib_signal_date'])).days if latest else np.nan)
 event_accession.append(latest['accession'] if latest else None);event_date.append(latest['first_conservative_Qlib_signal_date'] if latest else None)
frame=pd.DataFrame({'signal_date':cal,'NVDA_Data_Center_same_release_YoY':feature,'source_age_calendar_days':ages,'source_accession':event_accession,'source_effective_Qlib_date':event_date}).set_index('signal_date')
train=frame.loc['2011-01-03':'2022-12-30'];held=frame.loc['2023-01-03':'2026-10-09'];assert len(train)==3020 and len(held)==946
report={'decision':'original_SEC_NVDA_Data_Center_same_release_table_numeric_source_map_conditional_first_public_unqualified','plan_sha256':sha(card/'plan.json'),'selected_events':63,'numeric_events':len(numeric),'unknown_events':len(unknown),'first_numeric_filing_date':min((r['filing_date'] for r in numeric),default=None),'last_numeric_filing_date':max((r['filing_date'] for r in numeric),default=None),'numeric_by_year':{str(y):sum(r['announcement_year']==y for r in numeric) for y in range(2011,2027)},'unknown_by_status':{str(k):int(v) for k,v in pd.Series([r['status'] for r in unknown]).value_counts().items()},'training_rows':len(train),'training_known_days':int(train.NVDA_Data_Center_same_release_YoY.notna().sum()),'held_rows':len(held),'held_known_days':int(held.NVDA_Data_Center_same_release_YoY.notna().sum()),'source_age_max_known':int(frame.loc[frame.NVDA_Data_Center_same_release_YoY.notna(),'source_age_calendar_days'].max()) if len(numeric) else None,'definition_warning_events':sum(r['definition_warning_Mellanox'] for r in rows),'events':rows,'historical_first_issuer_release_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
static=pd.concat({'feature':frame[['NVDA_Data_Center_same_release_YoY']],'meta':frame[['source_age_calendar_days']]},axis=1);static.index=pd.MultiIndex.from_arrays([cal,['NVDA_DC']*len(cal)],names=['datetime','instrument']);ds=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2011-01-03','2022-12-30'),'held':('2023-01-03','2026-10-09')});assert len(ds.prepare('train',col_set=['feature','meta']))==3020 and len(ds.prepare('held',col_set=['feature','meta']))==946
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds);rec.save_objects(artifact_path='analysis',event_rows=rows,source_frame=frame,report={k:v for k,v in report.items() if k!='events'});rec.log_metrics(numeric_events=len(numeric),train_known=report['training_known_days'],held_known=report['held_known_days'],model_fits=0,native_accounts=0)
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'result':'result.json','model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Current and prior-year values are from same original CFO table, but report platform scope may change with acquisitions or taxonomy.','Conservative next-Qlib-day SEC acceptance proxy does not prove exact first issuer-public time.','No market outcome, incremental information or full account tested.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='nvda-dc-numeric-') as tmp:
 archive=Path(tmp)/'recorder.tar.gz'
 with archive.open('wb') as out:proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert proc.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/report') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'numeric_events':len(numeric),'unknown':report['unknown_by_status'],'first':report['first_numeric_filing_date'],'train_known':report['training_known_days'],'held_known':report['held_known_days'],'recorder_id':rid},indent=2))


# NVIDIA 原始 CFO 表的 Data Center 收入：首次数值与日历映射

从已核验的 63 份 NVIDIA 原始 SEC 季度申报，仅在 `EX-99.2` CFO commentary 的 `Revenue by Market Platform ($ in millions)` 表中读取精确 `Datacenter`/`Data Center` 行，保留同一版本的当前季、前季及上年同期金额，用当版金额计算同比，并与表内同比核对。没有从较晚申报向早期回填数值。按 SEC 接收日期后的**下一真实 Qlib 交易日**才使该事件对 Stage1 生效；这只是保守时点代理，不证明发行人首次公开时刻。

原生 Recorder 保存 63 个原始事件：**32 个**同版数值事件，最早 2016-05-12、最新 2026-08-26；21 个原件没有精确平台表行，另 **10 个**年度 Q4 附件同时含季度与全年同名行，本版保持歧义未知。日历映射的 2011–2022 训练 **1,284/3,020** 日、2023–2026 复用评价 **696/946** 日已知。2020 年起原件有 Mellanox 等业务范围变化，不能直接声称多年 Data Center 增长为恒定经济口径。

首次运行已将完整原始表事件和 3,966 行 Qlib 来源框架写入 FINISHED Recorder，随后仅在 `unknown_by_status` 的 NumPy 整数写 JSON 时失败。一次输出收尾从**同一 Recorder**读回已存的事件、框架与报告，规范化 JSON 标量，未重读 SEC 原件、未新建 Recorder 或修改数值；原失败与收尾收据均保留。Recorder `79535324617845bc9c5f75bb60c297b6` 的 3,020/946 行 DatasetH 和 Yanxuan NAS 哈希冷读通过。

独立复核确认 32 个同版同比计算及日历映射，但指出冻结计划要求保存**行、表、原文件三层哈希**；本版只保存了原文件哈希和精确单行单元格，缺行/表哈希。偏差见 `lineage_deviation.json`，数值不因此改变，却不能称表级来源身份完全封存。

本卡 0 SOXX 未来收益读取、0 拟合、0 账户。**决定：数值来源部分完成，但年度 Q4 歧义、表级哈希和历史首次公开时点未过门；暂不进入信息拟合。** 下一步在另一个有限预算下，按原始表标题限定修复 10 份年度附件并为接受行补齐行/表哈希；保留全年行与未决项，然后再检验与原 Stage1 39 项的增量及完整 0bp 账户。
